# 00 - Data Acquisition & Overview

This project forecasts daily store-level sales for Rossmann, a European drug store chain, using the **Rossmann Store Sales** dataset published on Kaggle (train.csv, test.csv, store.csv — 1,115 stores, ~2.5 years of daily history). Before any cleaning or feature engineering happens in later notebooks, this notebook's job is narrow and deliberate: load the raw CSVs exactly as Kaggle ships them, confirm their shape and schema line up with what the rest of the pipeline (`src/cleaning.py`, `src/features.py`) expects, quantify missingness in the store metadata, and explicitly identify the feature groups — temporal, promotional, store-level, and calendar/holiday — that the feature-engineering stage will build on. Nothing here is modeled yet; this is the acquisition-and-overview checkpoint the synopsis calls for.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

## Load the raw CSVs

We use the project's own `read_raw_csvs` helper (not a raw `pd.read_csv` call) so that the dtypes and parsed `Date` column are consistent with every other notebook in this pipeline. If the raw files aren't present yet, the `except` block below prints the exact steps needed to fetch them from Kaggle rather than failing silently.

In [ ]:
import pandas as pd

from src.data_io import read_raw_csvs, save_parquet

try:
    train, test, store = read_raw_csvs("../data/raw")
except FileNotFoundError as e:
    print(
        "Raw data not found in ../data/raw/.\n\n"
        "To fetch the Rossmann Store Sales dataset from Kaggle:\n"
        "  1. Create a Kaggle account (if you don't have one) and accept the competition rules at\n"
        "     https://www.kaggle.com/c/rossmann-store-sales/rules\n"
        "  2. Go to https://www.kaggle.com/settings/account and create a new API token\n"
        "     (this downloads a kaggle.json credentials file — place it at ~/.kaggle/kaggle.json)\n"
        "  3. From the project root, run:\n"
        "       kaggle competitions download -c rossmann-store-sales -p data/raw/\n"
        "  4. Unzip the downloaded archive inside data/raw/ so that train.csv, test.csv, and\n"
        "     store.csv sit directly in that folder.\n"
    )
    raise

## Shape and schema check

Before trusting the data, confirm row/column counts look sane (train should have roughly 1 million rows across 1,115 stores, test substantially fewer since it covers a short forecast horizon) and inspect each DataFrame's dtypes to make sure nothing was silently mis-parsed.

In [ ]:
print(f"train shape: {train.shape}")
print(f"test shape:  {test.shape}")
print(f"store shape: {store.shape}")

In [ ]:
train.info()

In [ ]:
store.info()

## Missingness in the store metadata

`store.csv` carries static per-store attributes (competition info, Promo2 enrollment) that are known to have gaps — e.g. not every store has an active competitor on record, and not every store participates in Promo2. We quantify this directly so we know what `src/cleaning.py`'s imputation step is actually compensating for.

In [ ]:
missing_pct = (store.isna().mean().sort_values(ascending=False) * 100).round(2)
missing_pct = missing_pct[missing_pct > 0]
missing_pct.to_frame(name="missing_%")

## Date coverage and store coverage

We confirm the train/test date ranges (and that they don't overlap — test should pick up right after train ends) and the number of distinct stores represented in train, which should match `store.csv`'s row count.

In [ ]:
print(f"train date range: {train['Date'].min().date()} to {train['Date'].max().date()}")
print(f"test date range:  {test['Date'].min().date()} to {test['Date'].max().date()}")
print(f"distinct stores in train: {train['Store'].nunique()}")

## Feature groups present in the raw data

This directly answers the synopsis objective — "identify relevant temporal, promotional, and store-level features" — by naming the four groups of raw columns that `src/features.py` will later expand into the full engineered feature matrix:

- **Temporal** — `DayOfWeek`, and the `Date` column itself, from which day/week/month/year and other calendar-derived signals (and later, lag/rolling-window features) are built.
- **Promotional** — `Promo` (whether a store is running a daily promotion), `Promo2` and `Promo2SinceWeek`/`Promo2SinceYear` (a secondary, longer-running promotion some stores opt into), and `PromoInterval` (which months Promo2 is active in).
- **Store-level** — `StoreType`, `Assortment` (product assortment breadth), and `CompetitionDistance`/`CompetitionOpenSinceMonth`/`CompetitionOpenSinceYear` (proximity and tenure of the nearest competitor).
- **Calendar/holiday** — `StateHoliday` (which holiday, if any) and `SchoolHoliday` (whether schools are closed), both of which shift demand independently of the weekly promo cycle.

Together these four groups are the raw material that `clean_pipeline` merges and imputes, and that `build_feature_matrix` turns into lag, rolling-window, and calendar features for modeling.

## Cache a typed copy of train

This is a convenience cache for faster re-loading during exploration in this notebook only — it is **not** a hard dependency for any downstream notebook, which all call `read_raw_csvs` (or `clean_pipeline`) directly on the raw CSVs.

In [ ]:
save_parquet(train, "../data/interim/train_raw_typed.parquet")